# Análise da Embalagens Aurora com pandas

3 meses de operação de uma indústria de embalagens: 100 clientes, 696 pedidos, 9 produtos.
A pergunta que guia a análise: **por que um mês despencou de faturamento, e onde o faturamento esconde pouca margem?**

Os dados estão na pasta `../dados/`. O dicionário de colunas está em `../dados/dicionario.md`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pedidos = pd.read_csv('../dados/pedidos.csv')
itens = pd.read_csv('../dados/itens_pedido.csv')
clientes = pd.read_csv('../dados/clientes.csv')
produtos = pd.read_csv('../dados/produtos.csv')

print(pedidos.shape, 'pedidos')
pedidos.head()

## 1. O mês que despencou

Agrupo o faturamento por mês pra achar o vale.

In [ ]:
pedidos['mes'] = pedidos['data_pedido'].str[:7]
fat_mes = pedidos.groupby('mes')['valor_total'].sum()
print(fat_mes)

fat_mes.plot(kind='bar', title='Faturamento por mês', color='#1F2D5C')
plt.ylabel('R$')
plt.tight_layout()
plt.show()

Abril é o vale (~R$ 6,56 mi) e junho recupera forte (~R$ 9,85 mi).

## 2. O faturamento engana

Margem líquida = valor - custo do produto - frete. Comparo faturamento com margem por
cliente (só quem fatura mais de R$ 300 mil) e ordeno pela margem, do pior pro melhor.

In [ ]:
pedidos['margem_liquida'] = (
    pedidos['valor_total'] - pedidos['custo_produto_total'] - pedidos['frete_real']
)

por_cliente = pedidos.groupby('cliente_codigo').agg(
    faturamento=('valor_total', 'sum'),
    margem=('margem_liquida', 'sum'),
)
por_cliente['margem_pct'] = (100 * por_cliente['margem'] / por_cliente['faturamento']).round(1)
por_cliente = por_cliente.merge(clientes[['cliente_codigo', 'nome', 'segmento']], on='cliente_codigo')

relevantes = por_cliente[por_cliente['faturamento'] > 300000]
relevantes.sort_values('margem_pct').head(8)[['nome', 'segmento', 'faturamento', 'margem_pct']]

Tem cliente que fatura alto e ainda assim aparece com margem espremida, alguns até
**negativa** (o frete e o custo comem o lucro). Faturar não é lucrar.

## 3. Mix de produto: qual variante rende mais

Cruzo os itens com o catálogo de produtos e vejo a margem por variante.

In [ ]:
itens_prod = itens.merge(produtos[['sku', 'variante']], on='sku')

por_variante = itens_prod.groupby('variante').agg(
    faturamento=('valor_linha', 'sum'),
    margem=('margem_linha', 'sum'),
)
por_variante['margem_pct'] = (100 * por_variante['margem'] / por_variante['faturamento']).round(1)
por_variante.sort_values('margem_pct', ascending=False)

A linha **Premium** rende a maior margem %, a **Padrão** a menor. Vender mais Premium
melhora a margem sem precisar vender mais volume.

## 4. Onde está o lucro por segmento

In [ ]:
ped_cli = pedidos.merge(clientes[['cliente_codigo', 'segmento']], on='cliente_codigo')

por_segmento = ped_cli.groupby('segmento').agg(
    faturamento=('valor_total', 'sum'),
    margem=('margem_liquida', 'sum'),
)
por_segmento['margem_pct'] = (100 * por_segmento['margem'] / por_segmento['faturamento']).round(1)
por_segmento.sort_values('faturamento', ascending=False)

## Conclusão

- Abril foi o vale de faturamento; junho recuperou.
- Faturar alto não garante margem: há clientes grandes com margem baixa ou negativa,
  puxada por custo e frete. É onde vale repensar preço ou frequência de atendimento.
- A variante Premium rende mais margem: mudar o mix de venda melhora o lucro.
- No segmento, alimentício fatura muito e ainda mantém boa margem; ecommerce e varejo
  faturam pouco e com margem espremida.

Ferramentas: pandas (`read_csv`, `groupby`, `agg`, `merge`), matplotlib.